# Credit-Card Fraud Detection — BiGAN and AnoGAN[cite: 1]
This notebook contains a functional implementation of BiGAN and AnoGAN for credit-card fraud detection[cite: 1]. No class definitions are used; all networks are built as `nn.Sequential` objects via factory functions and forwarded through plain functions[cite: 1].

## 1. Imports and Global Settings[cite: 1]
This cell imports necessary libraries, sets up global constants (like random seeds and file paths), and defines hyperparameters for both the BiGAN and AnoGAN architectures[cite: 1].

In [ ]:
import gc
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda iterable, **kw: iterable  # noqa: E731

# ─────────────────────────── Global settings ───────────────────────────
RAND_SEED = 42
DATA_FILE = "creditcard.csv"
HOLD_OUT_RATIO = 0.20
ARTIFACT_DIR = Path("results_plots")

# BiGAN hyper-parameters
BIGAN_Z_DIM = 16
BIGAN_ARCH = [128, 64]
BIGAN_PRE_EPOCHS = 20
BIGAN_PRE_LR = 1e-3
BIGAN_TRAIN_EPOCHS = 50
BIGAN_EG_LR = 1e-4
BIGAN_DISC_LR = 5e-5
BIGAN_MINI_BATCH = 512
BIGAN_SMOOTH = 0.05
BIGAN_REC_WT = 5.0
BIGAN_CYC_WT = 0.2
BIGAN_DROP = 0.10
BIGAN_DECAY = 1e-5

# AnoGAN hyper-parameters
ANOGAN_Z_DIM = 32
ANOGAN_ARCH = [64, 128, 64]
ANOGAN_PRE_EPOCHS = 20
ANOGAN_PRE_LR = 1e-3
ANOGAN_PRE_REC_WT = 1.0
ANOGAN_PRE_PRIOR_WT = 0.05
ANOGAN_PRE_DECAY = 1e-5
ANOGAN_TRAIN_EPOCHS = 80
ANOGAN_DISC_LR = 5e-5
ANOGAN_GEN_LR = 3e-4
ANOGAN_MINI_BATCH = 256
ANOGAN_SMOOTH = 0.2
ANOGAN_GP_WT = 10.0
ANOGAN_OPT_STEPS = 100
ANOGAN_OPT_LR = 5e-3
ANOGAN_OPT_ALPHA = 0.8

BLEND_GRID = np.array([0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95, 0.98, 0.99, 1.00])
COMPUTE_DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 2. Reproducibility and Device Setup[cite: 1]
This section contains utility functions to lock the random seeds across NumPy, Python's `random`, and PyTorch to ensure consistent results[cite: 1]. It also provides a helper to send tensors to the active compute device[cite: 1].

In [ ]:
def fix_randomness(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def worker_init(worker_id):
    base = torch.initial_seed() % (2 ** 32)
    np.random.seed(base)
    random.seed(base)

def send_to_device(tensor):
    return tensor.to(COMPUTE_DEV, non_blocking=(COMPUTE_DEV.type == "cuda"))


## 3. Data Pipeline[cite: 1]
These functions handle loading the CSV file, applying a log transformation to the 'Amount' column, splitting data, and scaling the features using `StandardScaler` (fitted only on training normals to prevent leakage)[cite: 1]. It also sets up PyTorch `DataLoader` objects[cite: 1].

In [ ]:
def load_csv_data(csv_path):
    print(f"Loading {csv_path} ...")
    raw = pd.read_csv(csv_path)
    raw["Amount"] = np.log1p(raw["Amount"].astype(np.float32))
    raw["Time"] = raw["Time"].astype(np.float32)
    feat = raw.drop(columns=["Class"]).to_numpy(dtype=np.float32)
    lbl = raw["Class"].astype(np.int64).to_numpy()
    print(f"  Records: {len(feat)}   Fraud: {lbl.sum()}")
    return feat, lbl

def build_train_test_arrays(feat, lbl):
    normal_rows = feat[lbl == 0]
    fraud_rows = feat[lbl == 1]

    train_normals, held_normals = train_test_split(
        normal_rows, test_size=HOLD_OUT_RATIO, random_state=RAND_SEED, shuffle=True
    )

    normaliser = StandardScaler().fit(train_normals)

    scaled_train = normaliser.transform(train_normals).astype(np.float32)
    scaled_held = normaliser.transform(held_normals).astype(np.float32)
    scaled_fraud = normaliser.transform(fraud_rows).astype(np.float32)

    scaled_test = np.vstack([scaled_held, scaled_fraud])
    test_lbl = np.concatenate([
        np.zeros(len(scaled_held), dtype=np.int64),
        np.ones(len(scaled_fraud), dtype=np.int64),
    ])
    return scaled_train, scaled_test, test_lbl

def make_loader(data_arr, mini_batch, rng_seed):
    rng = torch.Generator()
    rng.manual_seed(rng_seed)
    ds = TensorDataset(torch.from_numpy(data_arr))
    return DataLoader(
        ds,
        batch_size=mini_batch,
        shuffle=True,
        drop_last=True,
        worker_init_fn=worker_init,
        generator=rng,
        pin_memory=(COMPUTE_DEV.type == "cuda"),
    )


## 4. Network Factory Helpers[cite: 1]
This section defines factory functions to assemble Multi-Layer Perceptrons (MLPs) as `nn.Sequential` modules[cite: 1]. It includes definitions for the Encoders, Generators, and Discriminators used in both the BiGAN and AnoGAN architectures[cite: 1].

In [ ]:
def dense_sequence(dims, layernorm=False, drop_p=0.0):
    parts = []
    for k in range(len(dims) - 1):
        parts.append(nn.Linear(dims[k], dims[k + 1]))
        if k < len(dims) - 2:
            if layernorm:
                parts.append(nn.LayerNorm(dims[k + 1]))
            parts.append(nn.LeakyReLU(0.2, inplace=True))
            if drop_p > 0.0:
                parts.append(nn.Dropout(drop_p))
    return nn.Sequential(*parts)

def make_bigan_encoder(n_feat):
    return dense_sequence([n_feat] + BIGAN_ARCH + [BIGAN_Z_DIM], layernorm=True).to(COMPUTE_DEV)

def make_bigan_generator(n_feat):
    return dense_sequence([BIGAN_Z_DIM] + BIGAN_ARCH + [n_feat], layernorm=True).to(COMPUTE_DEV)

def make_bigan_discriminator(n_feat):
    body = dense_sequence(
        [n_feat + BIGAN_Z_DIM] + BIGAN_ARCH, layernorm=True, drop_p=BIGAN_DROP
    ).to(COMPUTE_DEV)
    head = nn.Sequential(nn.Linear(BIGAN_ARCH[-1], 1), nn.Sigmoid()).to(COMPUTE_DEV)
    return body, head

def bigan_disc_fwd(body, head, x_in, z_in, with_features=False):
    h = body(torch.cat([x_in, z_in], dim=1))
    p = head(h)
    return (p, h) if with_features else p

def make_anogan_generator(n_feat):
    return dense_sequence([ANOGAN_Z_DIM] + ANOGAN_ARCH + [n_feat]).to(COMPUTE_DEV)

def make_anogan_discriminator(n_feat):
    body = dense_sequence([n_feat] + ANOGAN_ARCH).to(COMPUTE_DEV)
    head = nn.Sequential(nn.Linear(ANOGAN_ARCH[-1], 1), nn.Sigmoid()).to(COMPUTE_DEV)
    return body, head

def anogan_disc_fwd(body, head, x_in, with_features=False):
    h = body(x_in)
    p = head(h)
    return (p, h) if with_features else p

def make_anogan_pre_encoder(n_feat):
    return dense_sequence([n_feat, 128, 64, ANOGAN_Z_DIM], layernorm=True).to(COMPUTE_DEV)


## 5. Loss Helpers[cite: 1]
Provides functions to compute the reconstruction loss (a mixture of Smooth L1 and MSE loss) and the gradient penalty used to stabilize adversarial training[cite: 1].

In [ ]:
def reconstruction_loss(x_hat, x_orig):
    return 0.7 * nn.functional.smooth_l1_loss(x_hat, x_orig) + 0.3 * nn.functional.mse_loss(x_hat, x_orig)

def gradient_penalty(body, head, real_x, fake_x):
    mix = torch.rand(real_x.size(0), 1, device=COMPUTE_DEV)
    interp = (mix * real_x + (1.0 - mix) * fake_x).requires_grad_(True)
    d_out = anogan_disc_fwd(body, head, interp)
    grads = torch.autograd.grad(
        d_out, interp,
        grad_outputs=torch.ones_like(d_out),
        create_graph=True, retain_graph=True,
    )[0]
    return ((grads.norm(2, dim=1) - 1.0) ** 2).mean()


## 6. BiGAN Training and Scoring[cite: 1]
Contains the warmup phase, the main adversarial training loop, and the scoring mechanism for the BiGAN model[cite: 1]. It handles encoder and generator updates alongside discriminator updates[cite: 1].

In [ ]:
def warmup_bigan(enc, gen, loader):
    ae_opt = optim.Adam(
        list(enc.parameters()) + list(gen.parameters()),
        lr=BIGAN_PRE_LR, weight_decay=BIGAN_DECAY,
    )
    print("  BiGAN encoder/generator warm-up ...")
    for ep in range(BIGAN_PRE_EPOCHS):
        enc.train(); gen.train()
        running = 0.0
        for (mb,) in loader:
            mb = send_to_device(mb)
            z = enc(mb)
            x_hat = gen(z)
            loss = reconstruction_loss(x_hat, mb)
            ae_opt.zero_grad(); loss.backward(); ae_opt.step()
            running += loss.item()
        print(f"    epoch {ep+1:02d}/{BIGAN_PRE_EPOCHS}  recon={running/len(loader):.4f}")

def train_bigan(train_arr, n_feat, rng_seed):
    enc = make_bigan_encoder(n_feat)
    gen = make_bigan_generator(n_feat)
    disc_body, disc_head = make_bigan_discriminator(n_feat)

    bce = nn.BCELoss()
    sl1 = nn.SmoothL1Loss()
    loader = make_loader(train_arr, BIGAN_MINI_BATCH, rng_seed)

    warmup_bigan(enc, gen, loader)

    eg_opt = optim.Adam(
        list(enc.parameters()) + list(gen.parameters()),
        lr=BIGAN_EG_LR, betas=(0.5, 0.999), weight_decay=BIGAN_DECAY,
    )
    disc_opt = optim.Adam(
        list(disc_body.parameters()) + list(disc_head.parameters()),
        lr=BIGAN_DISC_LR, betas=(0.5, 0.999), weight_decay=BIGAN_DECAY,
    )

    print("  BiGAN adversarial phase ...")
    for ep in range(BIGAN_TRAIN_EPOCHS):
        enc.train(); gen.train(); disc_body.train(); disc_head.train()
        d_sum = eg_sum = rec_sum = cyc_sum = 0.0

        for (mb,) in loader:
            mb = send_to_device(mb)
            n = mb.size(0)
            tgt_real = torch.full((n, 1), 1.0 - BIGAN_SMOOTH, device=COMPUTE_DEV)
            tgt_fake = torch.zeros(n, 1, device=COMPUTE_DEV)

            with torch.no_grad():
                z_noise = torch.randn(n, BIGAN_Z_DIM, device=COMPUTE_DEV)
                synth = gen(z_noise)
                enc_z = enc(mb)

            d_loss = (
                bce(bigan_disc_fwd(disc_body, disc_head, mb, enc_z.detach()), tgt_real)
                + bce(bigan_disc_fwd(disc_body, disc_head, synth.detach(), z_noise), tgt_fake)
            )
            disc_opt.zero_grad(); d_loss.backward(); disc_opt.step()

            enc_z = enc(mb)
            x_hat = gen(enc_z)
            z_noise = torch.randn(n, BIGAN_Z_DIM, device=COMPUTE_DEV)
            synth = gen(z_noise)
            z_cyc = enc(synth)

            rec_loss = reconstruction_loss(x_hat, mb)
            cyc_loss = sl1(z_cyc, z_noise)
            eg_loss = (
                bce(bigan_disc_fwd(disc_body, disc_head, mb, enc_z), tgt_fake)
                + bce(bigan_disc_fwd(disc_body, disc_head, synth, z_noise), tgt_real)
                + BIGAN_REC_WT * rec_loss
                + BIGAN_CYC_WT * cyc_loss
            )
            eg_opt.zero_grad(); eg_loss.backward(); eg_opt.step()

            d_sum += d_loss.item(); eg_sum += eg_loss.item()
            rec_sum += rec_loss.item(); cyc_sum += cyc_loss.item()

        nb = len(loader)
        print(
            f"    [BiGAN] {ep+1:02d}/{BIGAN_TRAIN_EPOCHS} "
            f"D={d_sum/nb:.4f} EG={eg_sum/nb:.4f} "
            f"rec={rec_sum/nb:.4f} cyc={cyc_sum/nb:.4f}"
        )

    return enc, gen, disc_body, disc_head

def compute_bigan_scores(enc, gen, disc_body, disc_head, data_arr, chunk=4096):
    for net in (enc, gen, disc_body, disc_head):
        net.eval()

    n = len(data_arr)
    res_buf = np.zeros(n, dtype=np.float32)
    disc_buf = np.zeros(n, dtype=np.float32)
    cyc_buf = np.zeros(n, dtype=np.float32)
    t_arr = torch.from_numpy(data_arr)

    with torch.no_grad():
        for start in tqdm(range(0, n, chunk), desc="BiGAN scoring"):
            seg = send_to_device(t_arr[start:start + chunk])
            z = enc(seg)
            x_hat = gen(z)
            z_hat = enc(x_hat)
            
            res_buf[start:start+chunk] = torch.mean(torch.abs(seg - x_hat), dim=1).cpu().numpy()

            _, fh_real = bigan_disc_fwd(disc_body, disc_head, seg, z, with_features=True)
            _, fh_rec = bigan_disc_fwd(disc_body, disc_head, x_hat, z, with_features=True)
            disc_buf[start:start+chunk] = torch.mean(torch.abs(fh_real - fh_rec), dim=1).cpu().numpy()

            cyc_buf[start:start+chunk] = torch.mean(torch.abs(z - z_hat), dim=1).cpu().numpy()

    auxiliary = (
        0.75 * iqr_normalise(disc_buf)
        + 0.25 * iqr_normalise(cyc_buf)
    )
    return res_buf, auxiliary


## 7. AnoGAN Training and Scoring[cite: 1]
This section implements the generator warmup, adversarial phase, and the latent-space optimization scoring process specific to the AnoGAN architecture[cite: 1].

In [ ]:
def warmup_anogan_gen(gen, train_arr, n_feat, rng_seed):
    pre_enc = make_anogan_pre_encoder(n_feat)
    pre_opt = optim.Adam(
        list(pre_enc.parameters()) + list(gen.parameters()),
        lr=ANOGAN_PRE_LR, weight_decay=ANOGAN_PRE_DECAY,
    )
    loader = make_loader(train_arr, ANOGAN_MINI_BATCH, rng_seed)

    print("  AnoGAN generator warm-up ...")
    for ep in range(ANOGAN_PRE_EPOCHS):
        pre_enc.train(); gen.train()
        t_sum = r_sum = p_sum = 0.0
        for (mb,) in loader:
            mb = send_to_device(mb)
            z = pre_enc(mb)
            x_hat = gen(z)
            rec = reconstruction_loss(x_hat, mb)
            mu_pen = torch.mean(z, dim=0).pow(2).mean()
            std_pen = (torch.std(z, dim=0, unbiased=False) - 1.0).pow(2).mean()
            prior = mu_pen + std_pen
            total = ANOGAN_PRE_REC_WT * rec + ANOGAN_PRE_PRIOR_WT * prior
            pre_opt.zero_grad(); total.backward(); pre_opt.step()
            t_sum += total.item(); r_sum += rec.item(); p_sum += prior.item()
        nb = len(loader)
        print(
            f"    epoch {ep+1:02d}/{ANOGAN_PRE_EPOCHS}  "
            f"total={t_sum/nb:.4f} rec={r_sum/nb:.4f} prior={p_sum/nb:.4f}"
        )

    del pre_enc
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def train_anogan(train_arr, n_feat, rng_seed):
    gen = make_anogan_generator(n_feat)
    disc_body, disc_head = make_anogan_discriminator(n_feat)

    bce = nn.BCELoss()
    g_opt = optim.Adam(gen.parameters(), lr=ANOGAN_GEN_LR, betas=(0.5, 0.999))
    disc_opt = optim.Adam(
        list(disc_body.parameters()) + list(disc_head.parameters()),
        lr=ANOGAN_DISC_LR, betas=(0.5, 0.999),
    )
    loader = make_loader(train_arr, ANOGAN_MINI_BATCH, rng_seed)

    warmup_anogan_gen(gen, train_arr, n_feat, rng_seed)

    print("  AnoGAN adversarial phase ...")
    for ep in range(ANOGAN_TRAIN_EPOCHS):
        gen.train(); disc_body.train(); disc_head.train()
        d_sum = g_sum = 0.0

        for (mb,) in loader:
            mb = send_to_device(mb)
            n = mb.size(0)
            tgt_real = torch.full((n, 1), 1.0 - ANOGAN_SMOOTH, device=COMPUTE_DEV)
            tgt_fake = torch.zeros(n, 1, device=COMPUTE_DEV)

            z_in = torch.randn(n, ANOGAN_Z_DIM, device=COMPUTE_DEV)
            fake_mb = gen(z_in).detach()
            d_loss = (
                bce(anogan_disc_fwd(disc_body, disc_head, mb), tgt_real)
                + bce(anogan_disc_fwd(disc_body, disc_head, fake_mb), tgt_fake)
            )
            if ANOGAN_GP_WT > 0:
                d_loss = d_loss + ANOGAN_GP_WT * gradient_penalty(disc_body, disc_head, mb, fake_mb)
            disc_opt.zero_grad(); d_loss.backward(); disc_opt.step()

            z_in = torch.randn(n, ANOGAN_Z_DIM, device=COMPUTE_DEV)
            g_loss = bce(
                anogan_disc_fwd(disc_body, disc_head, gen(z_in)),
                torch.ones(n, 1, device=COMPUTE_DEV),
            )
            g_opt.zero_grad(); g_loss.backward(); g_opt.step()

            d_sum += d_loss.item(); g_sum += g_loss.item()

        nb = len(loader)
        print(f"    [AnoGAN] {ep+1:02d}/{ANOGAN_TRAIN_EPOCHS} D={d_sum/nb:.4f} G={g_sum/nb:.4f}")

    return gen, disc_body, disc_head

def compute_anogan_scores(gen, disc_body, disc_head, data_arr, chunk=512):
    gen.eval(); disc_body.eval(); disc_head.eval()

    saved_states = []
    for net in (gen, disc_body, disc_head):
        st = [(p, p.requires_grad) for p in net.parameters()]
        for p, _ in st:
            p.requires_grad_(False)
        saved_states.append(st)

    n = len(data_arr)
    res_buf = np.zeros(n, dtype=np.float32)
    disc_buf = np.zeros(n, dtype=np.float32)
    t_arr = torch.from_numpy(data_arr)

    try:
        for start in tqdm(range(0, n, chunk), desc="AnoGAN scoring"):
            seg = send_to_device(t_arr[start:start + chunk])
            z_var = torch.randn(seg.size(0), ANOGAN_Z_DIM, device=COMPUTE_DEV, requires_grad=True)
            z_solver = optim.Adam([z_var], lr=ANOGAN_OPT_LR)

            with torch.no_grad():
                _, f_fixed = anogan_disc_fwd(disc_body, disc_head, seg, with_features=True)
            
            for _ in range(ANOGAN_OPT_STEPS):
                x_gen = gen(z_var)
                res_step = torch.mean(torch.abs(seg - x_gen), dim=1)
                _, f_gen = anogan_disc_fwd(disc_body, disc_head, x_gen, with_features=True)
                disc_step = torch.mean(torch.abs(f_fixed - f_gen), dim=1)
                obj = (ANOGAN_OPT_ALPHA * res_step + (1.0 - ANOGAN_OPT_ALPHA) * disc_step).sum()
                z_solver.zero_grad(); obj.backward(); z_solver.step()

            with torch.no_grad():
                x_gen = gen(z_var)
                res_final = torch.mean(torch.abs(seg - x_gen), dim=1)
                _, f_real2 = anogan_disc_fwd(disc_body, disc_head, seg, with_features=True)
                _, f_gen2 = anogan_disc_fwd(disc_body, disc_head, x_gen, with_features=True)
                disc_final = torch.mean(torch.abs(f_real2 - f_gen2), dim=1)

            res_buf[start:start+chunk] = res_final.cpu().numpy()
            disc_buf[start:start+chunk] = disc_final.cpu().numpy()
    finally:
        for net, st in zip((gen, disc_body, disc_head), saved_states):
            for p, req in st:
                p.requires_grad_(req)

    return res_buf, disc_buf


## 8. Score Utilities[cite: 1]
Helper functions for evaluating anomalies: normalizing scores via the Interquartile Range (IQR), fusing reconstruction and discriminator scores, and deriving the optimal precision-recall threshold[cite: 1].

In [ ]:
def iqr_normalise(arr):
    arr = np.asarray(arr, dtype=np.float32)
    med = np.median(arr)
    spread = np.percentile(arr, 75) - np.percentile(arr, 25)
    if spread < 1e-8:
        spread = np.std(arr) + 1e-8
    return (arr - med) / spread

def fuse_scores(res, disc, alpha):
    return alpha * iqr_normalise(res) + (1.0 - alpha) * iqr_normalise(disc)

def best_threshold(ground_truth, scores):
    prec, rec, cuts = precision_recall_curve(ground_truth, scores)
    if not len(cuts):
        return float(np.max(scores))
    f1_vals = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-10)
    return float(cuts[int(np.nanargmax(f1_vals))])


## 9. Evaluation[cite: 1]
Functions to evaluate the models against ground truth data, calculate metrics (Accuracy, Precision, Recall, F1, ROC/PR AUC), and prepare alpha-sweep tabular results[cite: 1].

In [ ]:
def evaluate_detector(ground_truth, res_scores, disc_scores, name):
    candidates = []
    for a in BLEND_GRID:
        s = fuse_scores(res_scores, disc_scores, a)
        cut = best_threshold(ground_truth, s)
        yhat = (s >= cut).astype(int)
        candidates.append({
            "alpha": float(a), "cut": cut, "scores": s, "preds": yhat,
            "Accuracy": accuracy_score(ground_truth, yhat),
            "Precision": precision_score(ground_truth, yhat, zero_division=0),
            "Recall": recall_score(ground_truth, yhat, zero_division=0),
            "F1": f1_score(ground_truth, yhat, zero_division=0),
            "ROC_AUC": roc_auc_score(ground_truth, s),
            "PR_AUC": average_precision_score(ground_truth, s),
        })

    best = max(candidates, key=lambda r: (r["F1"], r["Recall"], r["Precision"]))
    cm = confusion_matrix(ground_truth, best["preds"], labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    print(f"\n{'─'*18} {name} {'─'*18}")
    for k in ("alpha", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"):
        print(f"  {k:10s}: {best[k]:.4f}")
    print(f"  CM  TN={tn}  FP={fp}  FN={fn}  TP={tp}")

    summary = {
        "Model": name,
        "Accuracy": best["Accuracy"], "Precision": best["Precision"],
        "Recall": best["Recall"], "F1": best["F1"],
        "ROC_AUC": best["ROC_AUC"], "PR_AUC": best["PR_AUC"],
        "Alpha": best["alpha"], "Threshold": best["cut"],
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }
    best["tag"] = name
    return summary, cm, best

def alpha_sweep_table(ground_truth, res_scores, disc_scores):
    rows = []
    for a in BLEND_GRID:
        s = fuse_scores(res_scores, disc_scores, a)
        cut = best_threshold(ground_truth, s)
        yhat = (s >= cut).astype(int)
        rows.append({
            "alpha": float(a),
            "Accuracy": accuracy_score(ground_truth, yhat),
            "Precision": precision_score(ground_truth, yhat, zero_division=0),
            "Recall": recall_score(ground_truth, yhat, zero_division=0),
            "F1": f1_score(ground_truth, yhat, zero_division=0),
            "ROC_AUC": roc_auc_score(ground_truth, s),
            "PR_AUC": average_precision_score(ground_truth, s),
        })
    return pd.DataFrame(rows).sort_values("F1", ascending=False)


## 10. Plotting and Memory Management[cite: 1]
Utilities for generating visualizations (Confusion Matrices, Score Distributions, PR Curves) and saving them to disk, along with a function to free up GPU memory[cite: 1].

In [ ]:
def persist_figure(fig, fname):
    ARTIFACT_DIR.mkdir(exist_ok=True)
    fig.tight_layout()
    fig.savefig(ARTIFACT_DIR / fname, dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {ARTIFACT_DIR / fname}")

def plot_conf_matrices(ground_truth, champ_bi, champ_ano):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, champ in zip(axes, (champ_bi, champ_ano)):
        ConfusionMatrixDisplay.from_predictions(
            ground_truth, champ["preds"],
            display_labels=["Normal", "Fraud"],
            cmap="Blues", colorbar=False, ax=ax,
        )
        ax.set_title(f"{champ['tag']}  Confusion Matrix")
    persist_figure(fig, "conf_matrices.png")

def plot_score_histograms(ground_truth, champ_bi, champ_ano):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, champ in zip(axes, (champ_bi, champ_ano)):
        s = champ["scores"]
        ax.hist(s[ground_truth == 0], bins=60, alpha=0.65, density=True, label="Normal")
        ax.hist(s[ground_truth == 1], bins=60, alpha=0.65, density=True, label="Fraud")
        ax.axvline(champ["cut"], color="red", linestyle="--", label="Threshold")
        ax.set_title(f"{champ['tag']}  Score Distribution")
        ax.set_xlabel("Anomaly Score")
        ax.set_ylabel("Density")
        ax.legend()
    persist_figure(fig, "score_histograms.png")

def plot_pr_curves(ground_truth, champ_bi, champ_ano):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, champ in zip(axes, (champ_bi, champ_ano)):
        prec, rec, _ = precision_recall_curve(ground_truth, champ["scores"])
        ax.plot(rec, prec, lw=2)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
        ax.set_title(
            f"{champ['tag']}\n"
            f"PR-AUC={champ['PR_AUC']:.4f}   ROC-AUC={champ['ROC_AUC']:.4f}"
        )
    persist_figure(fig, "pr_curves.png")

def release_memory(*items):
    for it in items:
        del it
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 11. Main Execution[cite: 1]
The primary entry point that ties the notebook together: preparing data, sequentially training and evaluating both models, triggering plots, and writing metric summaries to CSV[cite: 1].

In [ ]:
def main():
    fix_randomness(RAND_SEED)
    print(f"Compute device: {COMPUTE_DEV}")

    feat, lbl = load_csv_data(DATA_FILE)
    n_feat = feat.shape[1]

    fix_randomness(RAND_SEED)
    train_arr, test_arr, test_lbl = build_train_test_arrays(feat, lbl)
    print(f"Train shape: {train_arr.shape}   Test shape: {test_arr.shape}   Fraud: {test_lbl.sum()}")

    # ── BiGAN ──
    print("\n=== BiGAN ===")
    enc, gen_bi, db_bi, dh_bi = train_bigan(train_arr, n_feat, RAND_SEED)
    bi_res, bi_disc = compute_bigan_scores(enc, gen_bi, db_bi, dh_bi, test_arr)
    bi_summary, bi_cm, bi_champ = evaluate_detector(test_lbl, bi_res, bi_disc, "BiGAN")
    bi_alpha_tbl = alpha_sweep_table(test_lbl, bi_res, bi_disc)
    release_memory(enc, gen_bi, db_bi, dh_bi, bi_res, bi_disc)

    # ── AnoGAN ──
    print("\n=== AnoGAN ===")
    gen_ano, db_ano, dh_ano = train_anogan(train_arr, n_feat, RAND_SEED)
    ano_res, ano_disc = compute_anogan_scores(gen_ano, db_ano, dh_ano, test_arr)
    ano_summary, ano_cm, ano_champ = evaluate_detector(test_lbl, ano_res, ano_disc, "AnoGAN")
    ano_alpha_tbl = alpha_sweep_table(test_lbl, ano_res, ano_disc)

    # ── Plots ──
    plot_conf_matrices(test_lbl, bi_champ, ano_champ)
    plot_score_histograms(test_lbl, bi_champ, ano_champ)
    plot_pr_curves(test_lbl, bi_champ, ano_champ)
    release_memory(gen_ano, db_ano, dh_ano, ano_res, ano_disc, train_arr, test_arr, test_lbl)

    # ── Final table ──
    result_df = pd.DataFrame([bi_summary, ano_summary])
    print("\n" + "=" * 64)
    print("BiGAN vs AnoGAN — Final Results")
    print("=" * 64)
    try:
        print(result_df.to_markdown(index=False))
    except ImportError:
        print(result_df.to_string(index=False))

    print(f"\nBiGAN CM:\n{bi_cm}")
    print(f"\nAnoGAN CM:\n{ano_cm}")

    result_df.to_csv("gan_detection_results.csv", index=False)
    bi_alpha_tbl.to_csv("bigan_alpha_sweep.csv", index=False)
    ano_alpha_tbl.to_csv("anogan_alpha_sweep.csv", index=False)
    print(f"\nArtifacts written to: gan_detection_results.csv, bigan_alpha_sweep.csv, anogan_alpha_sweep.csv, {ARTIFACT_DIR}/")

if __name__ == "__main__":
    main()
